In [1]:
import numpy as np
import pandas as pd

from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components.machines.compressor import Compressor
from caldis.components.combustion.heater import SimpleHeater
from caldis.fluids import CoolPropBackend
from caldis.solvers.solve import solve

import caldis
print("Caldis version:", getattr(caldis, "__version__", "unknown"))

Caldis version: 0.3.1


In [2]:
# --- fluide et conditions de référence ---
be = CoolPropBackend("Methane")
T0    = 25 + 273.15        # K  — ambiante / makeup / rejet coolers
P_low = 1.0e5              # Pa — basse pression (flash, makeup)
P_high = 100e5             # Pa — haute pression

# --- paramètres fixes ---
eta_comp     = 0.75        # rendement isentropique par étage
cooler_pinch = 5.0         # K  — coolers refroidissent vers T0
hex_pinch    = 5.0         # K  — pincement récupérateur (bout froid)
m_liq        = 1.0         # kg/s — débit de produit liquide (échelle)
max_ratio    = 5.0         # ratio de compression max par étage

T_cool_out = T0 + cooler_pinch     # ~30 °C en sortie de chaque cooler

# --- étagement de la compression (ratios égaux, <= max_ratio) ---
overall = P_high / P_low
n_stages = int(np.ceil(np.log(overall) / np.log(max_ratio)))
spr = overall ** (1.0 / n_stages)
pressures = [P_low * spr**i for i in range(n_stages + 1)]

print(f"{n_stages} étages, ratio/étage = {spr:.3f}")
for i, P in enumerate(pressures):
    print(f"  P{i+1} = {P/1e5:6.2f} bar")
P1, P2, P3, P4 = pressures

3 étages, ratio/étage = 4.642
  P1 =   1.00 bar
  P2 =   4.64 bar
  P3 =  21.54 bar
  P4 = 100.00 bar


In [3]:
# ====================================================================
#  L0 — étape 1 : train de compression intercoolé, ouvert
#  feed -> comp1 -> cooler1 -> comp2 -> cooler2 -> comp3 -> cooler3 -> sink HP
# ====================================================================

# --- composants ---
comp1 = Compressor("comp1", backend=be, eta_is=eta_comp)
comp2 = Compressor("comp2", backend=be, eta_is=eta_comp)
comp3 = Compressor("comp3", backend=be, eta_is=eta_comp)
cooler1 = SimpleHeater("cooler1", backend=be, Qdot=None, dp=0)
cooler2 = SimpleHeater("cooler2", backend=be, Qdot=None, dp=0)
cooler3 = SimpleHeater("cooler3", backend=be, Qdot=None, dp=0)
feed = FluidBoundary("feed", p=P1, T=T0, mdot=-m_liq, backend=be)   # makeup imposé
sink = FluidBoundary("sink", p=P4, backend=be)                      # aval HP, pression seule

# --- système ---
sys_ = System("L0_step1_compression")
sys_.add(feed, comp1, cooler1, comp2, cooler2, comp3, cooler3, sink)
sys_.connect(feed.port,   comp1.inl)
sys_.connect(comp1.out,   cooler1.inl)
sys_.connect(cooler1.out, comp2.inl)
sys_.connect(comp2.out,   cooler2.inl)
sys_.connect(cooler2.out, comp3.inl)
sys_.connect(comp3.out,   cooler3.inl)
sys_.connect(cooler3.out, sink.port)
sys_.assemble()

# --- contraintes : pressions de refoulement + températures sortie coolers ---
sys_.add_constraint(lambda: comp1.out.p - P2, "P_comp1")
sys_.add_constraint(lambda: comp2.out.p - P3, "P_comp2")
# sys_.add_constraint(lambda: comp3.out.p - P4, "P_comp3")
sys_.add_constraint(lambda: cooler1.out.T - T_cool_out, "T_cool1")
sys_.add_constraint(lambda: cooler2.out.T - T_cool_out, "T_cool2")
sys_.add_constraint(lambda: cooler3.out.T - T_cool_out, "T_cool3")

# --- points de départ (débit m_liq partout, ~430 K en sortie de compression) ---
SP = sys_.set_start_port
# SP(feed.port,   p=P1, T=T0,         mdot=-m_liq)
# SP(comp1.inl,   p=P1, T=T0,         mdot= m_liq);  SP(comp1.out,   p=P2, T=430, mdot=-m_liq)
# SP(cooler1.inl, p=P2, T=430,        mdot= m_liq);  SP(cooler1.out, p=P2, T=T_cool_out, mdot=-m_liq)
# SP(comp2.inl,   p=P2, T=T_cool_out, mdot= m_liq);  SP(comp2.out,   p=P3, T=430, mdot=-m_liq)
# SP(cooler2.inl, p=P3, T=430,        mdot= m_liq);  SP(cooler2.out, p=P3, T=T_cool_out, mdot=-m_liq)
# SP(comp3.inl,   p=P3, T=T_cool_out, mdot= m_liq);  SP(comp3.out,   p=P4, T=430, mdot=-m_liq)
# SP(cooler3.inl, p=P4, T=430,        mdot= m_liq);  SP(cooler3.out, p=P4, T=T_cool_out, mdot=-m_liq)
# SP(sink.port,   p=P4, T=T_cool_out, mdot= m_liq)
sys_.init_starts()

# --- résolution ---
sol = solve(sys_)

if sol and sol.success:
    W = sum(c.inl.mdot * (c.out.h - c.inl.h) for c in (comp1, comp2, comp3))
    print(f"\nW_comp total = {W/1e3:.1f} kW  (pour {abs(feed.port.mdot):.2f} kg/s)")
    for c in (comp1, comp2, comp3):
        print(f"  {c.local_name}: T_out = {be.T(c.out.p, c.out.h)-273.15:6.1f} °C")
    display(sys_.port_table())

✓ DOF: square (45 unknowns)
✓ start point: evaluable
⚠ 3 start-sanity warning(s):
   - comp1: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
   - comp2: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
   - comp3: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
  (non-blocking.)
  direct Newton: ok (7 it, |F_scaled|=1.4e-10)
✓ SOLVED (|F_scaled|=1.4e-10)

W_comp total = 1132.2 kW  (pour 1.00 kg/s)
  comp1: T_out =  178.8 °C
  comp2: T_out =  185.8 °C
  comp3: T_out =  188.6 °C


p_bar           T         T_C       h_kJkg   s_kJkgK  \
component port                                                              
feed      port    1.000000  298.150000   25.000000   909.965477  6.681318   
comp1     inl     1.000000  298.150000   25.000000   909.965477  6.681318   
          out     4.641589  451.957871  178.807871  1286.471413  6.898101   
cooler1   inl     4.641589  451.957871  178.807871  1286.471413  6.898101   
          out     4.641589  303.150000   30.000000   917.690583  5.914600   
comp2     inl     4.641589  303.150000   30.000000   917.690583  5.914600   
          out    21.544347  458.937758  185.787758  1298.483337  6.130280   
cooler2   inl    21.544347  458.937758  185.787758  1298.483337  6.130280   
          out    21.544347  303.150000   30.000000   901.437905  5.079523   
comp3     inl    21.544347  303.150000   30.000000   901.437905  5.079523   
          out   100.000000  461.714387  188.564387  1276.294765  5.289891   
cooler3   inl   100.000000  461.714387  188.564387  1276.294765  5.289891   
          out   100.000000  303.150000   30.000000   824.518530  4.091456   
sink      port  100.000000  303.150000   30.000000   824.518530  4.091456   

                  x        rho  mdot  
component port                        
feed      port  2.0   0.648277  -1.0  
comp1     inl   2.0   0.648277   1.0  
          out   2.0   1.983255  -1.0  
cooler1   inl   2.0   1.983255   1.0  
          out   2.0   2.976704  -1.0  
comp2     inl   2.0   2.976704   1.0  
          out   2.0   9.084108  -1.0  
cooler2   inl   2.0   9.084108   1.0  
          out   2.0  14.201509  -1.0  
comp3     inl   2.0  14.201509   1.0  
          out   2.0  41.923548  -1.0  
cooler3   inl   2.0  41.923548   1.0  
          out   2.0  73.837561  -1.0  
sink      port  2.0  73.837561   1.0

In [4]:
# ====================================================================
#  L0 — étape 2 : train de compression + récupérateur (ouvert)
#  Côté chaud : cooler3 -> hex.hot_in -> hex.hot_out -> sink_hot (HP)
#  Côté froid : cold_src (vapeur sat. 1 bar) -> hex.cold_in -> hex.cold_out -> cold_sink
#  Fermeture thermique : pincement bout froid (hot_out.T - cold_in.T = hex_pinch)
# ====================================================================
from caldis.components.HEX.heat_exchanger import SizingHX

m_cold = m_liq       # débit côté froid imposé pour ce test (≈ recyclage), à ajuster

# --- composants ---
comp1 = Compressor("comp1", backend=be, eta_is=eta_comp)
comp2 = Compressor("comp2", backend=be, eta_is=eta_comp)
comp3 = Compressor("comp3", backend=be, eta_is=eta_comp)
cooler1 = SimpleHeater("cooler1", backend=be, Qdot=None, dp=0)
cooler2 = SimpleHeater("cooler2", backend=be, Qdot=None, dp=0)
cooler3 = SimpleHeater("cooler3", backend=be, Qdot=None, dp=0)
hex = SizingHX("hex", backend_hot=be, backend_cold=be,
               h_hot=500, h_cold=500, area_ratio_cold_to_hot=1.0)

feed      = FluidBoundary("feed", p=P1, T=T0, mdot=-m_liq, backend=be)      # makeup HP side
sink_hot  = FluidBoundary("sink_hot", p=P4, backend=be)                     # aval chaud -> fixe la HP
cold_src  = FluidBoundary("cold_src", p=P1, h=be.h_pq(P1, 1.0),
                          mdot=-m_cold, backend=be)                         # vapeur sat. 1 bar imposée
cold_sink = FluidBoundary("cold_sink", backend=be)                         # aval froid : aucune spec

# --- système ---
sys_ = System("L0_step2_recuperator")
sys_.add(feed, comp1, cooler1, comp2, cooler2, comp3, cooler3,
         hex, sink_hot, cold_src, cold_sink)
# ligne chaude
sys_.connect(feed.port,    comp1.inl)
sys_.connect(comp1.out,    cooler1.inl)
sys_.connect(cooler1.out,  comp2.inl)
sys_.connect(comp2.out,    cooler2.inl)
sys_.connect(cooler2.out,  comp3.inl)
sys_.connect(comp3.out,    cooler3.inl)
sys_.connect(cooler3.out,  hex.hot_in)
sys_.connect(hex.hot_out,  sink_hot.port)
# ligne froide
sys_.connect(cold_src.port, hex.cold_in)
sys_.connect(hex.cold_out,  cold_sink.port)
sys_.assemble()

# --- contraintes ---
sys_.add_constraint(lambda: comp1.out.p - P2, "P_comp1")
sys_.add_constraint(lambda: comp2.out.p - P3, "P_comp2")
sys_.add_constraint(lambda: cooler1.out.T - T_cool_out, "T_cool1")
sys_.add_constraint(lambda: cooler2.out.T - T_cool_out, "T_cool2")
sys_.add_constraint(lambda: cooler3.out.T - T_cool_out, "T_cool3")
sys_.add_constraint(lambda: hex.hot_out.T - (-50+273.15), "pinch_hex")


SP = sys_.set_start_port
# --- côté chaud : p étagées, T~430 K post-compression / Tco post-cooler, débit m_liq ---
SP(feed.port,   p=P1, T=T0,         mdot=-m_liq)
SP(comp1.inl,   p=P1, T=T0,         mdot= m_liq);  SP(comp1.out,   p=P2, T=430, mdot=-m_liq)
SP(cooler1.inl, p=P2, T=430,        mdot= m_liq);  SP(cooler1.out, p=P2, T=T_cool_out, mdot=-m_liq)
SP(comp2.inl,   p=P2, T=T_cool_out, mdot= m_liq);  SP(comp2.out,   p=P3, T=430, mdot=-m_liq)
SP(cooler2.inl, p=P3, T=430,        mdot= m_liq);  SP(cooler2.out, p=P3, T=T_cool_out, mdot=-m_liq)
SP(comp3.inl,   p=P3, T=T_cool_out, mdot= m_liq);  SP(comp3.out,   p=P4, T=430, mdot=-m_liq)
SP(cooler3.inl, p=P4, T=430,        mdot= m_liq);  SP(cooler3.out, p=P4, T=T_cool_out, mdot=-m_liq)
# --- récupérateur (cible douce -50 °C côté chaud) ---
SP(hex.hot_in,  p=P4, T=T_cool_out,      mdot= m_liq)
SP(hex.hot_out, p=P4, T=273.15-50,       mdot=-m_liq)
SP(hex.cold_in, p=P1, h=be.h_pq(P1,1.0), mdot= m_cold)
SP(hex.cold_out,p=P1, T=273.15+0,        mdot=-m_cold)
# --- frontières : alignées sur leurs voisins ---
SP(sink_hot.port,  p=P4, T=273.15-50,       mdot= m_liq)
SP(cold_src.port,  p=P1, h=be.h_pq(P1,1.0), mdot=-m_cold)
SP(cold_sink.port, p=P1, T=273.15+0,        mdot= m_cold)
sys_.init_starts()

sol = solve(sys_, steps=20)


if sol and sol.success:
    print(f"\nhex: hot  {be.T(hex.hot_in.p,hex.hot_in.h)-273.15:6.1f} -> {be.T(hex.hot_out.p,hex.hot_out.h)-273.15:6.1f} °C")
    print(f"hex: cold {be.T(hex.cold_in.p,hex.cold_in.h)-273.15:6.1f} -> {be.T(hex.cold_out.p,hex.cold_out.h)-273.15:6.1f} °C")
    print(f"pincement bout froid = {be.T(hex.hot_out.p,hex.hot_out.h)-be.T(hex.cold_in.p,hex.cold_in.h):.1f} K")
    print(f"A_total = {hex.A_total:.2f} m²   UA = {hex.UA:.0f} W/K")
    display(sys_.port_table())

✓ DOF: square (65 unknowns)
✓ start point: evaluable
  direct Newton: ok (4 it, |F_scaled|=8.8e-09)
✓ SOLVED (|F_scaled|=8.8e-09)

hex: hot    30.0 ->  -50.0 °C
hex: cold -161.6 ->  -10.8 °C
pincement bout froid = 111.6 K
A_total = 36.48 m²   UA = 4560 W/K


p_bar           T         T_C       h_kJkg   s_kJkgK  \
component port                                                                  
feed      port        1.000000  298.150000   25.000000   909.965477  6.681318   
comp1     inl         1.000000  298.150000   25.000000   909.965477  6.681318   
          out         4.641589  451.957870  178.807870  1286.471412  6.898101   
cooler1   inl         4.641589  451.957870  178.807870  1286.471412  6.898101   
          out         4.641589  303.150000   30.000000   917.690583  5.914600   
comp2     inl         4.641589  303.150000   30.000000   917.690583  5.914600   
          out        21.544347  458.937758  185.787758  1298.483336  6.130280   
cooler2   inl        21.544347  458.937758  185.787758  1298.483336  6.130280   
          out        21.544347  303.150000   30.000000   901.437905  5.079523   
comp3     inl        21.544347  303.150000   30.000000   901.437905  5.079523   
          out       100.000000  461.714387  188.564387  1276.294764  5.289891   
cooler3   inl       100.000000  461.714387  188.564387  1276.294764  5.289891   
          out       100.000000  303.150000   30.000000   824.518530  4.091456   
hex       hot_in    100.000000  303.150000   30.000000   824.518530  4.091456   
          hot_out   100.000000  223.150000  -50.000000   503.657265  2.836448   
          cold_in     1.000000  111.507626 -161.642374   510.561947  4.578749   
          cold_out    1.000000  262.370087  -10.779913   831.423212  6.400782   
sink_hot  port      100.000000  223.150000  -50.000000   503.657265  2.836448   
cold_src  port        1.000000  111.507626 -161.642374   510.561947  4.578749   
cold_sink port        1.000000  262.370087  -10.779913   831.423212  6.400782   

                      x         rho  mdot  
component port                             
feed      port      2.0    0.648277  -1.0  
comp1     inl       2.0    0.648277   1.0  
          out       2.0    1.983255  -1.0  
cooler1   inl       2.0    1.983255   1.0  
          out       2.0    2.976704  -1.0  
comp2     inl       2.0    2.976704   1.0  
          out       2.0    9.084108  -1.0  
cooler2   inl       2.0    9.084108   1.0  
          out       2.0   14.201509  -1.0  
comp3     inl       2.0   14.201509   1.0  
          out       2.0   41.923548  -1.0  
cooler3   inl       2.0   41.923548   1.0  
          out       2.0   73.837561  -1.0  
hex       hot_in    2.0   73.837561   1.0  
          hot_out   2.0  175.209369  -1.0  
          cold_in   1.0    1.794607   1.0  
          cold_out  2.0    0.737399  -1.0  
sink_hot  port      2.0  175.209369   1.0  
cold_src  port      1.0    1.794607  -1.0  
cold_sink port      2.0    0.737399   1.0

In [5]:
from caldis.components.HEX.heat_exchanger import SizingHX
from caldis.solvers.autostart import autostart, fix_port

m_cold = m_liq     # au lieu de m_liq  (rééxécute toute la cellule step 2)

# --- composants ---
comp1 = Compressor("comp1", backend=be, eta_is=eta_comp)
comp2 = Compressor("comp2", backend=be, eta_is=eta_comp)
comp3 = Compressor("comp3", backend=be, eta_is=eta_comp)
cooler1 = SimpleHeater("cooler1", backend=be, Qdot=None, dp=0)
cooler2 = SimpleHeater("cooler2", backend=be, Qdot=None, dp=0)
cooler3 = SimpleHeater("cooler3", backend=be, Qdot=None, dp=0)
hex = SizingHX("hex", backend_hot=be, backend_cold=be, h_hot=500, h_cold=500, area_ratio_cold_to_hot=1.0)
feed      = FluidBoundary("feed", p=P1, T=T0, mdot=-m_liq, backend=be)
sink_hot  = FluidBoundary("sink_hot", p=P4, backend=be)
cold_src  = FluidBoundary("cold_src", p=P1, h=be.h_pq(P1, 1.0), mdot=-m_cold, backend=be)
cold_sink = FluidBoundary("cold_sink", backend=be)

sys_ = System("L0_step2_recuperator")
sys_.add(feed, comp1, cooler1, comp2, cooler2, comp3, cooler3, hex, sink_hot, cold_src, cold_sink)
sys_.connect(feed.port,   comp1.inl);   sys_.connect(comp1.out,  cooler1.inl)
sys_.connect(cooler1.out, comp2.inl);   sys_.connect(comp2.out,  cooler2.inl)
sys_.connect(cooler2.out, comp3.inl);   sys_.connect(comp3.out,  cooler3.inl)
sys_.connect(cooler3.out, hex.hot_in);  sys_.connect(hex.hot_out, sink_hot.port)
sys_.connect(cold_src.port, hex.cold_in); sys_.connect(hex.cold_out, cold_sink.port)
sys_.assemble()

# --- contraintes "porteuses de valeur" via fix_port (posent l'équation ET ancrent le départ) ---
fix_port(sys_, comp1.out, p=P2)
fix_port(sys_, comp2.out, p=P3)
fix_port(sys_, cooler1.out, T=T_cool_out)
fix_port(sys_, cooler2.out, T=T_cool_out)
fix_port(sys_, cooler3.out, T=T_cool_out)
# relation arbitraire -> reste un add_constraint classique
sys_.add_constraint(lambda: hex.hot_in.T - hex.cold_out.T - hex_pinch, "pinch_hex")

# --- seuls hints restants : 2 températures non linéaires sur le hex (anti-DTLM) ---
# sys_.set_start_port(hex.hot_out,  T=273.15 - 50)
# sys_.set_start_port(hex.cold_out, T=273.15 + 20)

autostart(sys_, verbose=True)     # remplace init_starts()
sol = solve(sys_, steps=20)

if sol and sol.success:
    print(f"\nhex: hot  {be.T(hex.hot_in.p,hex.hot_in.h)-273.15:6.1f} -> {be.T(hex.hot_out.p,hex.hot_out.h)-273.15:6.1f} °C")
    print(f"hex: cold {be.T(hex.cold_in.p,hex.cold_in.h)-273.15:6.1f} -> {be.T(hex.cold_out.p,hex.cold_out.h)-273.15:6.1f} °C")
    display(sys_.port_table())

autostart: 38 propagated, 0 pressure fallbacks, 10 port-quantities left at default
✓ DOF: square (65 unknowns)
✓ start point: evaluable
  direct Newton: ok (5 it, |F_scaled|=6.5e-10)
✓ SOLVED (|F_scaled|=6.5e-10)

hex: hot    30.0 ->  -61.8 °C
hex: cold -161.6 ->   25.0 °C


p_bar           T         T_C       h_kJkg   s_kJkgK  \
component port                                                                  
feed      port        1.000000  298.150000   25.000000   909.965477  6.681318   
comp1     inl         1.000000  298.150000   25.000000   909.965477  6.681318   
          out         4.641589  451.957871  178.807871  1286.471413  6.898101   
cooler1   inl         4.641589  451.957871  178.807871  1286.471413  6.898101   
          out         4.641589  303.150000   30.000000   917.690583  5.914600   
comp2     inl         4.641589  303.150000   30.000000   917.690583  5.914600   
          out        21.544347  458.937758  185.787758  1298.483337  6.130280   
cooler2   inl        21.544347  458.937758  185.787758  1298.483337  6.130280   
          out        21.544347  303.150000   30.000000   901.437905  5.079523   
comp3     inl        21.544347  303.150000   30.000000   901.437905  5.079523   
          out       100.000000  461.714387  188.564387  1276.294765  5.289891   
cooler3   inl       100.000000  461.714387  188.564387  1276.294765  5.289891   
          out       100.000000  303.150000   30.000000   824.518530  4.091456   
hex       hot_in    100.000000  303.150000   30.000000   824.518530  4.091456   
          hot_out   100.000000  211.331746  -61.818254   425.115001  2.474852   
          cold_in     1.000000  111.507626 -161.642374   510.561947  4.578749   
          cold_out    1.000000  298.150000   25.000000   909.965477  6.681318   
sink_hot  port      100.000000  211.331746  -61.818254   425.115001  2.474852   
cold_src  port        1.000000  111.507626 -161.642374   510.561947  4.578749   
cold_sink port        1.000000  298.150000   25.000000   909.965477  6.681318   

                      x         rho  mdot  
component port                             
feed      port      2.0    0.648277  -1.0  
comp1     inl       2.0    0.648277   1.0  
          out       2.0    1.983255  -1.0  
cooler1   inl       2.0    1.983255   1.0  
          out       2.0    2.976704  -1.0  
comp2     inl       2.0    2.976704   1.0  
          out       2.0    9.084108  -1.0  
cooler2   inl       2.0    9.084108   1.0  
          out       2.0   14.201509  -1.0  
comp3     inl       2.0   14.201509   1.0  
          out       2.0   41.923548  -1.0  
cooler3   inl       2.0   41.923548   1.0  
          out       2.0   73.837561  -1.0  
hex       hot_in    2.0   73.837561   1.0  
          hot_out   2.0  224.067282  -1.0  
          cold_in   1.0    1.794607   1.0  
          cold_out  2.0    0.648277  -1.0  
sink_hot  port      2.0  224.067282   1.0  
cold_src  port      1.0    1.794607  -1.0  
cold_sink port      2.0    0.648277   1.0

In [6]:
from caldis.components.HEX.heat_exchanger import SizingHX
from caldis.components.valves.valve import Valve
from caldis.components.vessels.flash_tank import FlashTank
from caldis.solvers.autostart import autostart, fix_port

# --- components ---
comp1 = Compressor("comp1", backend=be, eta_is=eta_comp)
comp2 = Compressor("comp2", backend=be, eta_is=eta_comp)
comp3 = Compressor("comp3", backend=be, eta_is=eta_comp)
cooler1 = SimpleHeater("cooler1", backend=be, Qdot=None, dp=0)
cooler2 = SimpleHeater("cooler2", backend=be, Qdot=None, dp=0)
cooler3 = SimpleHeater("cooler3", backend=be, Qdot=None, dp=0)
hex = SizingHX("hex", backend_hot=be, backend_cold=be, h_hot=500, h_cold=500, area_ratio_cold_to_hot=1.0)
val = Valve("val", backend=be)                      # sizing : Kv inconnu, dp fixé par les pressions
flash_sep = FlashTank("flash_sep", backend=be)
feed      = FluidBoundary("feed", p=P1, T=T0, mdot=-m_liq, backend=be)   # makeup imposé
product   = FluidBoundary("product",  backend=be)                   # soutirage liquide : p seule
cold_sink = FluidBoundary("cold_sink", p=P1, backend=be)                 # vapeur sortante : ancre la BP

# --- system ---
sys_ = System("L0_step3_valve_flash")
sys_.add(feed, comp1, cooler1, comp2, cooler2, comp3, cooler3, hex, val, flash_sep, product, cold_sink)
sys_.connect(feed.port,    comp1.inl);  sys_.connect(comp1.out, cooler1.inl)
sys_.connect(cooler1.out,  comp2.inl);  sys_.connect(comp2.out, cooler2.inl)
sys_.connect(cooler2.out,  comp3.inl);  sys_.connect(comp3.out, cooler3.inl)
sys_.connect(cooler3.out,  hex.hot_in); sys_.connect(hex.hot_out, val.inl)
sys_.connect(val.out,      flash_sep.inl)
sys_.connect(flash_sep.liq, product.port)
sys_.connect(flash_sep.vap, hex.cold_in); sys_.connect(hex.cold_out, cold_sink.port)
sys_.assemble()

# --- value-bearing constraints via fix_port ---
fix_port(sys_, comp1.out, p=P2)
fix_port(sys_, comp2.out, p=P3)
fix_port(sys_, comp3.out, p=P4)
fix_port(sys_, cooler1.out, T=T_cool_out)
fix_port(sys_, cooler2.out, T=T_cool_out)
fix_port(sys_, cooler3.out, T=T_cool_out)

sys_.add_constraint(lambda: hex.hot_in.T - hex.cold_out.T - hex_pinch, "pinch_hex")


# # legitimate nonlinear hints: seed the cold branch (propagation can't derive these)
# sys_.set_start_port(hex.hot_out,  T=273.15 - 150)       # deeply cooled HP gas before the valve
# sys_.set_start_port(hex.cold_in,  h=be.h_pq(P1, 1.0))   # saturated vapour from the flash
# sys_.set_start_port(hex.cold_out, T=273.15 - 20)        # warmed recycle leaving the recuperator


# hf = be.h_pq(P1, 0.0); hg = be.h_pq(P1, 1.0)
# sys_.set_start_port(flash_sep.liq, h=hf, mdot=-m_liq/10)   # saturated liquid
# sys_.set_start_port(flash_sep.vap, h=hg, mdot=-m_liq)      # saturated vapour

sys_.init_starts()

autostart(sys_, verbose=True)


sol = solve(sys_, steps=40)

if sol and sol.success:
    x = be.quality(flash_sep.inl.p, flash_sep.inl.h)
    print(f"\nflash inlet x    = {x:.3f}")
    print(f"liquide produit  = {-flash_sep.liq.mdot:.4f} kg/s   (fraction {-flash_sep.liq.mdot/m_liq*100:.1f} % du makeup)")
    print(f"vapeur recyclee  = {-flash_sep.vap.mdot:.4f} kg/s")
    print(f"hex hot   {be.T(hex.hot_in.p,hex.hot_in.h)-273.15:6.1f} -> {be.T(hex.hot_out.p,hex.hot_out.h)-273.15:6.1f} degC")
    print(f"hex cold  {be.T(hex.cold_in.p,hex.cold_in.h)-273.15:6.1f} -> {be.T(hex.cold_out.p,hex.cold_out.h)-273.15:6.1f} degC")
    print(f"Kv vanne = {val.Kv:.3e}")
    display(sys_.port_table())

autostart: 40 propagated, 0 pressure fallbacks, 22 port-quantities left at default
✓ DOF: square (78 unknowns)
✓ start point: evaluable
⚠ 3 start-sanity warning(s):
   - hex: cold inlet/outlet nearly equal -> cold MASS FLOW is undetermined (anchor the cold-side inlet/outlet TEMPERATURES)
   - flash_sep: vap mdot should be < 0 (outlet)
   - flash_sep: liq mdot should be < 0 (outlet)
  (non-blocking.)
  direct Newton: ok (6 it, |F_scaled|=6.5e-09)
✓ SOLVED (|F_scaled|=6.5e-09)

flash inlet x    = 0.906
liquide produit  = 0.0938 kg/s   (fraction 9.4 % du makeup)
vapeur recyclee  = 0.9062 kg/s
hex hot     30.0 ->  -56.1 degC
hex cold  -161.6 ->   25.0 degC
Kv vanne = 3.178e-04


p_bar           T         T_C       h_kJkg   s_kJkgK  \
component port                                                                  
feed      port        1.000000  298.150000   25.000000   909.965477  6.681318   
comp1     inl         1.000000  298.150000   25.000000   909.965477  6.681318   
          out         4.641589  451.957871  178.807871  1286.471413  6.898101   
cooler1   inl         4.641589  451.957871  178.807871  1286.471413  6.898101   
          out         4.641589  303.150000   30.000000   917.690583  5.914600   
comp2     inl         4.641589  303.150000   30.000000   917.690583  5.914600   
          out        21.544347  458.937758  185.787758  1298.483337  6.130280   
cooler2   inl        21.544347  458.937758  185.787758  1298.483337  6.130280   
          out        21.544347  303.150000   30.000000   901.437905  5.079523   
comp3     inl        21.544347  303.150000   30.000000   901.437905  5.079523   
          out       100.000000  461.714387  188.564387  1276.294765  5.289891   
cooler3   inl       100.000000  461.714387  188.564387  1276.294765  5.289891   
          out       100.000000  303.150000   30.000000   824.518530  4.091456   
hex       hot_in    100.000000  303.150000   30.000000   824.518530  4.091456   
          hot_out   100.000000  217.008792  -56.141208   462.596558  2.649851   
          cold_in     1.000000  111.507626 -161.642374   510.561947  4.578749   
          cold_out    1.000000  298.150000   25.000000   909.965477  6.681318   
val       inl       100.000000  217.008792  -56.141208   462.596558  2.649851   
          out         1.000000  111.507626 -161.642374   462.596558  4.148596   
flash_sep inl         1.000000  111.507626 -161.642374   462.596558  4.148596   
          vap         1.000000  111.507626 -161.642374   510.561947  4.578749   
          liq         1.000000  111.507626 -161.642374    -0.557339 -0.004967   
product   port        1.000000  111.507626 -161.642374    -0.557339 -0.004967   
cold_sink port        1.000000  298.150000   25.000000   909.965477  6.681318   

                               x         rho      mdot  
component port                                          
feed      port      2.000000e+00    0.648277 -1.000000  
comp1     inl       2.000000e+00    0.648277  1.000000  
          out       2.000000e+00    1.983255 -1.000000  
cooler1   inl       2.000000e+00    1.983255  1.000000  
          out       2.000000e+00    2.976704 -1.000000  
comp2     inl       2.000000e+00    2.976704  1.000000  
          out       2.000000e+00    9.084108 -1.000000  
cooler2   inl       2.000000e+00    9.084108  1.000000  
          out       2.000000e+00   14.201509 -1.000000  
comp3     inl       2.000000e+00   14.201509  1.000000  
          out       2.000000e+00   41.923548 -1.000000  
cooler3   inl       2.000000e+00   41.923548  1.000000  
          out       2.000000e+00   73.837561 -1.000000  
hex       hot_in    2.000000e+00   73.837561  1.000000  
          hot_out   2.000000e+00  200.054595 -1.000000  
          cold_in   1.000000e+00    1.794607  0.906156  
          cold_out  2.000000e+00    0.648277 -0.906156  
val       inl       2.000000e+00  200.054595  1.000000  
          out       9.061562e-01    1.979590 -1.000000  
flash_sep inl       9.061562e-01    1.979590  1.000000  
          vap       1.000000e+00    1.794607 -0.906156  
          liq       1.467243e-10  422.588475 -0.093844  
product   port      1.467243e-10  422.588475  0.093844  
cold_sink port      2.000000e+00    0.648277  0.906156

In [9]:
from caldis.components.fittings.mixer import Mixer
hf = be.h_pq(P1, 0.0); hg = be.h_pq(P1, 1.0)

comp1 = Compressor("comp1", backend=be, eta_is=eta_comp)
comp2 = Compressor("comp2", backend=be, eta_is=eta_comp)
comp3 = Compressor("comp3", backend=be, eta_is=eta_comp)
cooler1 = SimpleHeater("cooler1", backend=be, Qdot=None, dp=0)
cooler2 = SimpleHeater("cooler2", backend=be, Qdot=None, dp=0)
cooler3 = SimpleHeater("cooler3", backend=be, Qdot=None, dp=0)
mixer = Mixer("mixer", backend=be)
hex = SizingHX("hex", backend_hot=be, backend_cold=be, h_hot=500, h_cold=500, area_ratio_cold_to_hot=1.0)
val = Valve("val", backend=be)
flash_sep = FlashTank("flash_sep", backend=be)
feed    = FluidBoundary("feed", p=P1, T=T0, backend=be)       # makeup : p,T imposes, debit libre
product = FluidBoundary("product", mdot=m_liq, backend=be)    # production imposee

sys_ = System("L0_step4_closed")
sys_.add(feed, comp1, cooler1, comp2, cooler2, comp3, cooler3, mixer, hex, val, flash_sep, product)
sys_.connect(feed.port,   mixer.in1)
sys_.connect(mixer.out,   comp1.inl)
sys_.connect(comp1.out,   cooler1.inl); sys_.connect(cooler1.out, comp2.inl)
sys_.connect(comp2.out,   cooler2.inl); sys_.connect(cooler2.out, comp3.inl)
sys_.connect(comp3.out,   cooler3.inl); sys_.connect(cooler3.out, hex.hot_in)
sys_.connect(hex.hot_out, val.inl);     sys_.connect(val.out, flash_sep.inl)
sys_.connect(flash_sep.liq, product.port)
sys_.connect(flash_sep.vap, hex.cold_in); sys_.connect(hex.cold_out, mixer.in2)
sys_.assemble()

fix_port(sys_, comp1.out, p=P2)
fix_port(sys_, comp2.out, p=P3)
fix_port(sys_, comp3.out, p=P4)
fix_port(sys_, cooler1.out, T=T_cool_out)
fix_port(sys_, cooler2.out, T=T_cool_out)
fix_port(sys_, cooler3.out, T=T_cool_out)
sys_.add_constraint(lambda: hex.hot_in.T - hex.cold_out.T - hex_pinch, "pinch_hex")  # warm end

# nonlinear + circulation start hints
sys_.set_start_port(feed.port,     mdot=-m_liq)         # start only: makeup ~ production
sys_.set_start_port(hex.hot_out,   T=273.15 - 50)
sys_.set_start_port(hex.cold_in,   h=hg)
sys_.set_start_port(hex.cold_out,  T=273.15 + 25)
sys_.set_start_port(flash_sep.liq, h=hf, mdot=-m_liq)
sys_.set_start_port(flash_sep.vap, h=hg, mdot=-9*m_liq)
# val.set_internal_start("Kv", 3e-3)     # circulation ~10 kg/s, dp~99 bar -> Kv ~ 3e-3

autostart(sys_, verbose=True)
sol = solve(sys_, steps=20)

autostart: 53 propagated, 0 pressure fallbacks, 8 port-quantities left at default
✓ DOF: square (84 unknowns)
✓ start point: evaluable
  direct Newton: ok (7 it, |F_scaled|=9.4e-10)
✓ SOLVED (|F_scaled|=9.4e-10)
